In [22]:
import pandas as pd
import random
from pathlib import Path
import os, shutil, pathlib
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
def data_present(data_path, kind):
    df = pd.read_csv(data_path)
    print(f"\nAnalyzing {kind} data")
    for col in ["label", "labels", "category", "diagnosis", "target", "class"]:
        if col in df.columns:
            counts = df[col].value_counts().sort_index()
            print(f"Number of unique labels: {counts.size}")
            print(counts)
            break
    else:
        if df.shape[1] >= 2:
            col = df.columns[1]
            counts = df[col].value_counts().sort_index()
            print(f"\nUsing column '{col}' as label (fallback).")
            print(f"Number of unique labels: {counts.size}")
            print(counts)
        else:
            print("No obvious label column found in train_split.csv")
    return df

In [24]:
TRAIN_SPLIT_CSV = "./data/train_split.csv"
df_train = data_present(TRAIN_SPLIT_CSV, "train")


Analyzing train data
Number of unique labels: 12
labels
complex                            1432
frog_eye_leaf_spot                 2869
frog_eye_leaf_spot complex          151
healthy                            4156
powdery_mildew                     1067
powdery_mildew complex               83
rust                               1668
rust complex                         86
rust frog_eye_leaf_spot             108
scab                               4352
scab frog_eye_leaf_spot             621
scab frog_eye_leaf_spot complex     175
Name: count, dtype: int64


In [25]:
TEST_SPLIT_CSV = "./data/test_split.csv"
df_test = data_present(TEST_SPLIT_CSV, "test")


Analyzing test data
Number of unique labels: 12
labels
complex                            170
frog_eye_leaf_spot                 312
frog_eye_leaf_spot complex          14
healthy                            468
powdery_mildew                     117
powdery_mildew complex               4
rust                               192
rust complex                        11
rust frog_eye_leaf_spot             12
scab                               474
scab frog_eye_leaf_spot             65
scab frog_eye_leaf_spot complex     25
Name: count, dtype: int64


In [26]:
class_map = ["healthy", "complex",  "frog_eye_leaf_spot", "rust", "scab", "powdery_mildew"]

In [27]:
def copy_data(data_class):
    OUT_DIR = f"./data/{data_class.lower()}"
    if os.path.exists(OUT_DIR):
        shutil.rmtree(OUT_DIR)
    os.makedirs(OUT_DIR, exist_ok=True)

    label_col = next((c for c in df_train.columns if c in ["label","labels","category","diagnosis","target","class"]), None)
    if label_col is None:
        if df_train.shape[1] >= 2:
            label_col = df_train.columns[1]
        else:
            raise SystemExit("Cant find label column, please specify label_col manually")

    file_col = next((c for c in df_train.columns if any(k in c.lower() for k in ["path","file","image","filename","filepath","audio"])), None)
    if file_col is None:
        file_col = df_train.columns[0] if df_train.columns[0] != label_col else (df_train.columns[1] if df_train.shape[1] > 1 else None)
        if file_col is None:
            raise SystemExit("cant find file column, please specify file_col manually")

    # if needed, set the base directory where the files are located
    BASE_DIR = "./data/train_images"
    
    rows = df_train[df_train[label_col].astype(str).str.lower().str.contains(data_class.lower())]
    print(f"Found {len(rows)} rows labeled {data_class} (label column: {label_col}, file column: {file_col})")

    missing = []
    copied = 0
    for _, r in rows.iterrows():
        src = r[file_col]
        src_path = pathlib.Path(src)
        if not src_path.is_absolute():
            src_path = pathlib.Path(BASE_DIR) / src if BASE_DIR else src_path
        if src_path.exists():
            dst = pathlib.Path(OUT_DIR) / src_path.name
            shutil.copy2(src_path, dst)
            copied += 1
        else:
            missing.append(str(src_path))

    print(f"Copied {copied} files to {OUT_DIR}")
    if missing:
        print(f"{len(missing)} files were missing. Examples:", missing[:10])

In [28]:
# for i in class_map:
#     copy_data(i)

In [29]:
def data_split(data_source, gen_dir, data_albe, ratio = 0.8):
    SRC = data_source
    DST_TRAIN = gen_dir / f"train{data_albe}"
    DST_TEST  = gen_dir / f"test{data_albe}"
    RANDOM_SEED = 42
    SPLIT_RATIO = ratio

    if not SRC.exists() or not SRC.is_dir():
        raise SystemExit(f"Source folder not found: {SRC}")

    files = [p for p in SRC.iterdir() if p.is_file()]
    n = len(files)
    if n == 0:
        raise SystemExit(f"No files found in {SRC}")

    for d in (DST_TRAIN, DST_TEST):
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)

    random.seed(RANDOM_SEED)
    random.shuffle(files)
    split_idx = int(n * SPLIT_RATIO)
    train_files = files[:split_idx]
    test_files  = files[split_idx:]

    for p in train_files:
        shutil.copy2(p, DST_TRAIN / p.name)
    for p in test_files:
        shutil.copy2(p, DST_TEST / p.name)

    print(f"Total files: {n}")
    print(f"Train ({len(train_files)}): {DST_TRAIN}")
    print(f"Test  ({len(test_files)}): {DST_TEST}")

In [30]:
def data_gen(disease_kind):
    healthy_source = Path("./data/healthy")
    disease_source = Path(f"./data/{disease_kind.lower()}")
    gen_dir = Path(f"./data/healthy2{disease_kind.lower()}")
    data_split(healthy_source, gen_dir, "A", ratio=0.8)
    data_split(disease_source, gen_dir, "B", ratio=0.8)

In [31]:
for i in class_map[1:]:
    data_gen(i)

Total files: 4156
Train (3324): data/healthy2complex/trainA
Test  (832): data/healthy2complex/testA
Total files: 1927
Train (1541): data/healthy2complex/trainB
Test  (386): data/healthy2complex/testB
Total files: 4156
Train (3324): data/healthy2frog_eye_leaf_spot/trainA
Test  (832): data/healthy2frog_eye_leaf_spot/testA
Total files: 3924
Train (3139): data/healthy2frog_eye_leaf_spot/trainB
Test  (785): data/healthy2frog_eye_leaf_spot/testB
Total files: 4156
Train (3324): data/healthy2rust/trainA
Test  (832): data/healthy2rust/testA
Total files: 1862
Train (1489): data/healthy2rust/trainB
Test  (373): data/healthy2rust/testB
Total files: 4156
Train (3324): data/healthy2scab/trainA
Test  (832): data/healthy2scab/testA
Total files: 5148
Train (4118): data/healthy2scab/trainB
Test  (1030): data/healthy2scab/testB
Total files: 4156
Train (3324): data/healthy2powdery_mildew/trainA
Test  (832): data/healthy2powdery_mildew/testA
Total files: 1150
Train (920): data/healthy2powdery_mildew/trainB